# Lecture 20: A/B Testing and Randomized Experiments
v.ekc

The permutation test, end to end, then to show causality, we add **random assignment**. Rejecting the null supports a *causal* conclusion.

**Today's sections:**
1. Comparing two samples (A/B testing)
2. Random permutation (shuffling)
3. Simulation under the null
4. The permutation test
5. Try It, a randomized controlled experiment

In [ ]:
from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

---
## 1. Comparing Two Samples (A/B Testing)

New question type: do babies of smoking mothers weigh less? Two groups: smokers and non-smokers. Are the two distributions *really* different, or is the gap just sampling noise?

In [ ]:
births = Table.read_table('baby.csv')

In [ ]:
births

For today, just the smoking label and the birth weight:

In [ ]:
smoking_and_birthweight = births.select('Maternal Smoker', 'Birth Weight')

In [ ]:
smoking_and_birthweight.group('Maternal Smoker')

Overlaid histograms, the picture version of the question:

In [ ]:
smoking_and_birthweight.hist('Birth Weight', group='Maternal Smoker')

### Test Statistic

[Question] What values of our statistic are in favor of the alternative: positive or negative?

In [ ]:
means_table = smoking_and_birthweight.group('Maternal Smoker', np.average)
means_table

In [ ]:
means = means_table.column(1)
observed_difference = means.item(1) - means.item(0)
observed_difference

Package the computation into a function (Lab 7 will thank you):

In [ ]:
def difference_of_means(table, label, group_label):
    """Takes: name of table, column label of numerical variable,
    column label of group-label variable
    Returns: Difference of means of the two groups"""
    
    #table with the two relevant columns
    reduced = table.select(label, group_label)  
    
    # table containing group means
    means_table = reduced.group(group_label, np.average)
    # array of group means
    means = means_table.column(1)
    
    return means.item(1) - means.item(0)

In [ ]:
difference_of_means(births, 'Birth Weight', 'Maternal Smoker')

### Try It 1: A different gap 😊

Smokers' babies weigh less. Do smoking and non-smoking mothers differ in **age** too?

1. Draw the overlaid histograms of `Maternal Age` for the two groups.

2. Use `difference_of_means` to compute the observed age gap (smokers minus non-smokers).

3. Eyeball verdict: does the age gap look as convincing as the weight gap?

In [ ]:
# 1. overlaid age histograms


In [ ]:
# 2. observed age difference


*3. Your interpretation: (double-click to edit)*

<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*The function takes any numerical column; that is why we wrote it.*

```python
# 1.
births.hist('Maternal Age', group='Maternal Smoker')

# 2.
difference_of_means(births, 'Maternal Age', 'Maternal Smoker')
```

*3. Smokers average about 0.8 years younger, and the histograms overlap heavily. Far less dramatic than the 9.27-ounce weight gap. Small does not automatically mean chance, though: Try It 4 runs the permutation test on it.*

</details>

---
## 2. Random Permutation (Shuffling)

**The null says the label doesn't matter**, so let's literally shuffle the labels and see what differences pure chance produces. `sample(with_replacement=False)` is a shuffle.

In [ ]:
paper_people = make_array('Michael', 'Dwight', 'Angela', 'Phyllis', 'Stanley', 'Oscar', 'Jim','Pam','Toby')
job = make_array('best boss', 'assistant regional manager', 'accounting', 'accounting','accounting','accounting','sales','front desk','hr')

office = Table().with_columns("people person's paper people", paper_people, "job", job)

In [ ]:
office.sample()

Sampling *without* replacement uses every row exactly once. That is a shuffle:

In [ ]:
office.sample(with_replacement = False)

In [ ]:
# job trade!!
office.with_column('Shuffled', office.sample(with_replacement = False).column(0))

---
## 3. Simulation Under the Null

Attach shuffled labels, recompute the difference of means: that's one draw from the null world.

In [ ]:
smoking_and_birthweight

In [ ]:
shuffled_labels = smoking_and_birthweight.sample(with_replacement=False).column('Maternal Smoker')

Attach the shuffled labels next to the real ones:

In [ ]:
original_and_shuffled = smoking_and_birthweight.with_column(
    'Shuffled Label', shuffled_labels
)

In [ ]:
original_and_shuffled

Difference of means with the shuffled labels, then with the real ones:

In [ ]:
# test statistic under shuffled labels
#    under null hypothesis conditions

difference_of_means(original_and_shuffled, 'Birth Weight', 'Shuffled Label')

In [ ]:
# observed test statistic

difference_of_means(original_and_shuffled, 'Birth Weight', 'Maternal Smoker')

### Try It 2: Shuffle by hand 😊

1. In one cell, produce a fresh null-world difference yourself: shuffle the labels, attach them, and compute the difference of means. Run it three times and watch the values.

2. Re-running the cell above (real labels) never changes its answer. Why not?

In [ ]:
# 1. your own shuffled difference


*2. Your interpretation: (double-click to edit)*

<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Only the shuffled column is random; the real labels never move.*

```python
# 1.
new_shuffle = smoking_and_birthweight.sample(with_replacement=False).column('Maternal Smoker')
shuffled_tbl = smoking_and_birthweight.with_column('Shuffled Label', new_shuffle)
difference_of_means(shuffled_tbl, 'Birth Weight', 'Shuffled Label')
```

*2. The real labels are fixed data: the same 1,174 rows every run, so the same two group means. Typical shuffled differences stay within about plus or minus 3 ounces, nowhere near the real 9.27.*

</details>

---
## 4. The Permutation Test

Under the null hypothesis, we assume the labels do not matter. So, we will use a Permutation Test. To implement a permutation test, we shuffle the labels and then calculate the test statistic for each group. If our observed test statistic lies within the expected range of our simulated test statistics, that is support for the null model!

If not, then the alternative model might be better.

Repeat the shuffle 2500 times and compare the observed difference to the shuffled ones:

In [ ]:
def one_simulated_difference(table, label, group_label):
    """Takes: name of table, column label of numerical variable,
    column label of group-label variable
    Returns: Difference of means of the two groups after shuffling labels"""
    
    # array of shuffled labels
    shuffled_labels = table.sample(with_replacement = False).column(group_label)
    
    # table of numerical variable and shuffled labels
    shuffled_table = table.select(label).with_column('Shuffled Label', shuffled_labels)
    
    return difference_of_means(shuffled_table, label, 'Shuffled Label')   

In [ ]:
one_simulated_difference(births, 'Birth Weight', 'Maternal Smoker')

Repeat the shuffle 2500 times to map the null world:

In [ ]:
differences = make_array()

for i in np.arange(2500):
    new_difference = one_simulated_difference(births, 'Birth Weight', 'Maternal Smoker')
    differences = np.append(differences, new_difference)

In [ ]:
Table().with_column('Difference Between Group Means', differences).hist()
print('Observed Difference:', observed_difference)
plots.title('Prediction Under the Null Hypothesis');
plots.scatter(observed_difference, -0.01, color='red', s=120);

### Try It 3: The p-value 😊

1. Compute the p-value: the fraction of the 2500 shuffled differences that are *as extreme or more* than `observed_difference`. (Which direction counts as extreme here?)

2. Your conclusion about smoking and birth weight, in hypothesis-test language.

In [ ]:
# 1. p-value


*2. Your interpretation: (double-click to edit)*

<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Small (very negative) differences favor the alternative, so count the low tail.*

```python
# 1.
np.count_nonzero(differences <= observed_difference) / len(differences)
```

*2. p = 0.0: not one of 2500 shuffles came near the observed gap. Reject the null; the two groups' birth weights genuinely differ. (Whether smoking *causes* the gap is a separate question: keep reading.)*

</details>

> **Read the picture:** the observed difference (−9.27 oz) is nowhere near the shuffled differences (all within ±5). The null ("smoking label doesn't matter") is untenable. Lab 7 (Great British Bake Off) is this exact test with cake. 🍰

> **p-value = 0.** Not one of 2500 shuffles produced a difference as extreme as the real one. But note: mothers *chose* whether to smoke, so this shows a real difference, not yet a cause. For causation we need the next section.

### Try It 4: Test the age gap 😊

In Try It 1 you saw smoking mothers average about 0.8 years younger. Time to test it. **Null:** the age distributions of the two groups are the same. **Alternative:** smokers are younger.

1. Compute the observed difference in average `Maternal Age` (smokers minus non-smokers).

2. Simulate 1000 shuffled age differences and draw their histogram. (`one_simulated_difference` does the work.)

3. Compute the p-value and give your conclusion. Is a *small* gap the same thing as a *chance* gap?

In [ ]:
# 1. observed age difference


In [ ]:
# 2. 1000 shuffled age differences


In [ ]:
# 3. p-value


*3. Your conclusion: (double-click to edit)*

<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Same machinery, new column: that is why we wrote functions.*

```python
# 1.
observed_age_diff = difference_of_means(births, 'Maternal Age', 'Maternal Smoker')
observed_age_diff

# 2.
age_diffs = make_array()
for i in np.arange(1000):
    age_diffs = np.append(age_diffs, one_simulated_difference(births, 'Maternal Age', 'Maternal Smoker'))
Table().with_column('Shuffled Age Difference', age_diffs).hist()

# 3.
np.count_nonzero(age_diffs <= observed_age_diff) / 1000
```

*3. The p-value lands around 0.01, so the age gap, small as it is, is more than chance: reject the null. A small effect and a chance effect are different things. (And mothers were not randomly assigned to smoke, so this is a real difference, not yet a cause.)*

</details>

---
## 5. Try It: A Randomized Controlled Experiment 💉

31 chronic back-pain patients were **randomly assigned**: 15 to botulinum toxin A, 16 to saline placebo. `Result` = 1 means pain relief.

In [ ]:
botox = Table.read_table('bta.csv')
botox.show()

### Try It 5: Botox or placebo 😊

1. In one line of code: how many people in each group experienced pain relief (and how many didn't)?

2. In one line of code: what *proportion* of each group experienced relief?

In [ ]:
# Fill in: replace the ... 😅
...

In [ ]:
# Fill in: replace the ... 😅
...

<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Counts want a pivot; proportions of a 0/1 column are just averages.*

```python
# 1.
botox.pivot('Result', 'Group')

# 2.
botox.group('Group', np.average)     # Control: 0.125, Treatment: 0.6
```

</details>

> **Why random assignment matters:** because a coin, not the patients, decided who got botox, the two groups differ *only* by treatment and chance. So when a permutation test rejects "chance," what's left is the treatment: a **causal** conclusion. That's the entire logic of clinical trials, and of Homework 6's final question.

---
> **Today's takeaway:** shuffle to simulate the null, compare to the observed gap, and if assignment was random, a rejected null means the treatment *works*.

## Appendix — Quick Reference

Full `datascience` docs: [data8.org/datascience](https://data8.org/datascience/)

### Minimal template — A/B test (permutation test)
```python
# A/B test, answered with a permutation test
# 1. Observed statistic
observed = difference_of_means(t, value_label, group_label)

# 2. Simulate under the null: shuffle the labels, recompute, repeat
differences = make_array()
for i in np.arange(2500):
    shuffled = t.sample(with_replacement=False).column(group_label)
    one_diff = difference_of_means(t.with_column('Shuffled', shuffled), value_label, 'Shuffled')
    differences = np.append(differences, one_diff)

# 3. p-value: fraction of shuffles at least as extreme as observed
#    (direction comes from the alternative; here, smaller = more extreme)
p_value = np.count_nonzero(differences <= observed) / len(differences)

# 4. Conclusion: small p-value → the groups differ.
#    Causal only if the groups were randomly assigned (RCT).
```